# Camera Server / Client Example

This notebook shows the normal camera workflow through the ZMQ camera server. It starts the server, connects a client, reads frames from shared memory, changes camera settings, tests ROI resizing, and shuts everything down cleanly.

In [ ]:
# Core libraries
import time

import matplotlib.pyplot as plt
import numpy as np

import pwi_inst.hardware.Cameras.Camera_Client as CamClientlib
import pwi_inst.hardware.Cameras.Camera_Server as CamServerlib
import pwi_inst.hardware.Cameras.Camera_Viewer as CamViewerlib
from pwi_inst.hardware.Cameras.Camera_Widget import CameraWidget

plt.style.use("dark_background")

In [ ]:
%load_ext autoreload
%aimport pwi_inst.hardware.Cameras.FLIRPointGreyCameras.flir_flycapture2_ctypes
%aimport pwi_inst.hardware.Cameras.FLIRPointGreyCameras.FLIR_PointGrey
%aimport pwi_inst.hardware.Cameras.Camera_Client
%aimport pwi_inst.hardware.Cameras.Camera_Server
%aimport pwi_inst.hardware.Cameras.Camera_Viewer
%aimport pwi_inst.hardware.Cameras.Camera_Widget
%autoreload 1

## Configuration

In [ ]:
HOST = "127.0.0.1"
COMMAND_PORT = 50731
FRAME_PUB_PORT = 50732

CAMERA_TYPE = "FLIR Point Grey"
CAMERA_KWARGS = {
    "CameraIdx": 0,
    "verbose": True,
}

POLL_SLEEP = 0.001

## Start The Camera Server

In [ ]:
# Stop a previous server created by this notebook, if one is still around.
try:
    camserver.stopProcess()
    time.sleep(0.5)
except NameError:
    pass

camserver = CamServerlib.CameraZMQServer(
    host=HOST,
    command_port=COMMAND_PORT,
    frame_pub_port=FRAME_PUB_PORT,
    CameraType=CAMERA_TYPE,
    CameraKwargs=CAMERA_KWARGS,
    PollSleep=POLL_SLEEP,
)

camserver.startProcess()
time.sleep(1.0)

print(f"Server alive: {camserver.Process.is_alive()}")

## Connect A Client

In [ ]:
try:
    client.close()
except NameError:
    pass

client = CamClientlib.CameraClient(
    host=HOST,
    command_port=COMMAND_PORT,
    frame_pub_port=FRAME_PUB_PORT,
    timeout_ms=5000,
    client_id="camera_notebook",
)

properties = client.GetProperties()
properties

## Read And Display A Frame

In [ ]:
def show_frame(frame, title=None):
    plt.figure(figsize=(7, 6))

    if frame.ndim == 2:
        plt.imshow(frame, cmap="gray")
    else:
        plt.imshow(frame)

    if title is not None:
        plt.title(title)

    plt.axis("off")
    plt.show()


frame = client.GetFrame(WaitForNewFrame=True)
print(f"Frame shape: {frame.shape}, dtype: {frame.dtype}")
show_frame(frame, "Latest frame")

## Basic Camera Controls

In [ ]:
client.SetExposureTime(2)
client.SetGain(0.5)

print(f"Exposure: {client.GetExposureTime()}")
print(f"Gain: {client.GetGain()}")

## ROI Resize Test

In [ ]:
print("Current ROI:", client.GetROI())

roi_result = client.SetROI(864, 608, 200, 200)
print("Set ROI result:", roi_result)
print("New ROI:", client.GetROI())

frame = client.GetFrame(WaitForNewFrame=True)
print(f"ROI frame shape: {frame.shape}, dtype: {frame.dtype}")
show_frame(frame, "ROI frame")

In [ ]:
# Restore full-frame acquisition.
client.SetROI(enable=False)
print("Restored ROI:", client.GetROI())

frame = client.GetFrame(WaitForNewFrame=True)
print(f"Full frame shape: {frame.shape}, dtype: {frame.dtype}")
show_frame(frame, "Restored full frame")

## Trigger Modes

In [ ]:
client.SetContinuousMode()
print("Continuous trigger mode:", client.GetTriggerMode())

# For software-triggered acquisition, uncomment these lines.
# client.SetSoftwareTriggerMode()
# frame = client.GetSoftwareTriggeredFrame()
# show_frame(frame, "Software-triggered frame")

## Optional Viewer Process

In [ ]:
viewer = CamViewerlib.CameraViewer(
    host=HOST,
    command_port=COMMAND_PORT,
    frame_pub_port=FRAME_PUB_PORT,
    initial_scale=1,
)
viewer.startProcess()

## Optional Control Widget

In [ ]:
cam_widget = CameraWidget(
    host=HOST,
    command_port=COMMAND_PORT,
    frame_pub_port=FRAME_PUB_PORT,
)
cam_widget.startProcess()

## Cleanup

In [ ]:
# Close the client handles when you are done with this notebook.
try:
    client.close()
except NameError:
    pass

# Stop the server when no viewers or widgets are still connected.
try:
    camserver.stopProcess()
except NameError:
    pass